# SAC arrival_v2 — Cross extension (600k → 1M) + P1 v6 (1.5M)

**前情**：`sac_arrival_v2_cross_u10_regression_completed.ipynb` 在 600k 时 `final_success=0.867`（gate 阈值 0.85 PASS），但 `last100k_mean=0.6533 < 0.78`（gate FAIL）。Eval 曲线 500k→600k 单调上升（0.60→0.63→0.70→0.87），同时 `path_efficiency` 还在涨（0.57→0.75）、`eval_time` 还在掉（109s→65s）。判断为「未收敛」非「奖励学坏」。

**本 notebook 任务**：

| Phase | 任务 | benchmark | flow | obs | total_steps | 预算 |
|---|---|---|---|---|---:|---:|
| 1 | cross extension (resume from 600k) | `single_u10_cross_tgt15` | `wake_v8_U1p00_Re150` | s0 / 10-D | 600k → **1M** | ~1h L4 |
| 2 | P1 v6 (新启) | `single_u15_upstream_tgt15` | `wake_v8_U1p50_Re250` | s1 / 12-D | 0 → **1.5M** | ~6h L4 |

**Phase 1 gate**：`final ≥ 0.85` AND `last100k_mean ≥ 0.9 × peak` AND `final_oob_rate ≤ 0.10`，且 obs/replay 语义为 `arrival_v2` 设计。

**Phase 2 准入**：Phase 1 gate 全过。否则 Phase 2 自动 hold，回头改 reward / 调超参。

**输出**：
- Phase 1（与 600k run 同 path，append 续训）：`experiments/arrival_v2_prototype/cross_u10_regression/arrival_v2/sac_vanilla/s0_k4/seed_46/`
- Phase 2：`experiments/arrival_v2_prototype/p1_v6/arrival_v2/sac_vanilla/s1_k4/seed_46/`

**风格说明**：训练全部用 `!python -u -m scripts.train_sac` 直接调用（不再用 `%%bash` cell magic），Colab 内可实时滚动看到 train log。

## 0. GPU sanity

In [ ]:
!nvidia-smi | head -10
import torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}  '
      f'device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')

## 1. Mount Drive + cwd

和 600k run 用同一个 working clone（确保 commit `813096e` arrival_v2 实现 + `online_sac_reward_redesign.md` 修订）。

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO_DIR = '/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'
%cd $REPO_DIR

## 2. 通用配置 — Phase 1 (cross extension) + Phase 2 (P1 v6)

Phase 1 与 600k run 完全同 path / 同 hyperparam，仅 `total_steps` 600k → 1M（`--resume` 路径自动 pick up state）。

Phase 2 是新启 run，预算 1.5M（cross 需要 ~600-1000k 才稳，upstream + s1 + 12-D 比 cross 难，留缓冲）。

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

# ==== Phase 1 — cross extension (resume 600k → 1M) ====
BENCHMARK_KEY = 'single_u10_cross_tgt15'
FLOW_PATH = 'wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy'
TASK_GEOMETRY = 'cross_stream'
TARGET_SPEED = 1.5
OBJECTIVE = 'arrival_v2'
PROBE_LAYOUT = 's0'
HISTORY_LENGTH = 4
SEED = 46

TOTAL_STEPS = 1_000_000              # 600k → 1M (净增 400k)
RANDOM_STEPS = 5_000
UPDATE_AFTER = 5_000
BATCH_SIZE = 256
HIDDEN_DIM = 256
NUM_ENVS = 6
EVAL_EVERY = 25_000
EVAL_EPISODES = 30
CHECKPOINT_EVERY = 100_000
DEVICE = 'cuda'

PASS_FINAL_SUCCESS = 0.85
PASS_LAST100_RATIO = 0.90
PASS_OOB_RATE = 0.10

RUN_ROOT = Path('experiments/arrival_v2_prototype/cross_u10_regression/arrival_v2/sac_vanilla/s0_k4/seed_46')
CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/cross_u10_regression/arrival_v2/sac_vanilla/s0_k4/seed_46')
MANIFEST_PATH = Path(f'benchmarks/{BENCHMARK_KEY}.json')
RUN_ROOT_STR = str(RUN_ROOT)
CKPT_ROOT_STR = str(CKPT_ROOT)
MANIFEST_PATH_STR = str(MANIFEST_PATH)

# ==== Phase 2 — P1 v6 (1.5M, 新启) ====
P1_BENCHMARK_KEY = 'single_u15_upstream_tgt15'
P1_FLOW_PATH = 'wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy'
P1_PROBE_LAYOUT = 's1'
P1_TASK_GEOMETRY = 'upstream'

P1_TOTAL_STEPS = 1_500_000           # 比 cross 难,留缓冲;若提前稳定,看 trainer_state 早停

P1_RUN_ROOT = Path('experiments/arrival_v2_prototype/p1_v6/arrival_v2/sac_vanilla/s1_k4/seed_46')
P1_CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/p1_v6/arrival_v2/sac_vanilla/s1_k4/seed_46')
P1_MANIFEST_PATH = Path(f'benchmarks/{P1_BENCHMARK_KEY}.json')
P1_RUN_ROOT_STR = str(P1_RUN_ROOT)
P1_CKPT_ROOT_STR = str(P1_CKPT_ROOT)
P1_MANIFEST_PATH_STR = str(P1_MANIFEST_PATH)

os.environ['PYTHONUNBUFFERED'] = '1'

print('---- Phase 1 (cross extension) ----')
print(f'  benchmark    = {BENCHMARK_KEY}')
print(f'  flow         = {FLOW_PATH}')
print(f'  total_steps  = 600,000 → {TOTAL_STEPS:,} (resume + extend)')
print(f'  run_root     = {RUN_ROOT}')
print(f'  ckpt_root    = {CKPT_ROOT}')
print()
print('---- Phase 2 (P1 v6) ----')
print(f'  benchmark    = {P1_BENCHMARK_KEY}')
print(f'  flow         = {P1_FLOW_PATH}')
print(f'  probe        = {P1_PROBE_LAYOUT} / {P1_TASK_GEOMETRY}')
print(f'  total_steps  = {P1_TOTAL_STEPS:,}')
print(f'  run_root     = {P1_RUN_ROOT}')
print(f'  ckpt_root    = {P1_CKPT_ROOT}')

## 3. Preflight — flow / arrival_v2 candidate gate / reward unit tests / manifest

停止条件：
- Phase 1 flow 缺失 → 直接 raise；P1 flow 缺失只 warn（Phase 2 跑前再 hard check）
- `validate_arrival_v2_candidate` 失败 → arrival_v2 reward 公式不满足 discounted unsafe-shortcut / terminal dominance
- `test_reward_objective.py` 失败 → reward 实现退化
- manifest 生成失败 → eval 不可重复

In [ ]:
flow_path = Path(FLOW_PATH)
if not flow_path.exists():
    raise FileNotFoundError(f'missing Phase 1 flow file: {flow_path}')
print(f'[OK] Phase 1 flow exists: {flow_path} ({flow_path.stat().st_size / 1e6:.1f} MB)')

p1_flow_path = Path(P1_FLOW_PATH)
if not p1_flow_path.exists():
    print(f'[warn] Phase 2 flow not present yet: {p1_flow_path}')
    print('       Phase 2 cell 会在 hold check 时再 raise，可先把 cross extension 跑掉')
else:
    print(f'[OK] Phase 2 flow exists: {p1_flow_path} ({p1_flow_path.stat().st_size / 1e6:.1f} MB)')

In [ ]:
!python -u -m scripts.validate_arrival_v2_candidate

In [ ]:
!python -u -m pytest tests/test_reward_objective.py -q

In [ ]:
!python -u -m scripts.generate_standard_benchmarks --benchmarks {BENCHMARK_KEY} --episodes {EVAL_EPISODES}
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f'manifest not generated: {MANIFEST_PATH}')
print(f'[OK] Phase 1 manifest ready: {MANIFEST_PATH}')

## 4. Phase 1 — Cross extension 600k → 1M (resume)

`--resume` 会从 `trainer_state.json` 自动读出原 600k run 的全部 config（reward objective、obs context、timeout terminal、num_envs、seed、reward params 等），CLI 上的 default arg 不会覆盖；只有显式传的 `--total-steps 1_000_000` 会替换原 600k。

Skip 逻辑：若 `trainer_state.env_step >= TOTAL_STEPS` 就跳过（重跑 notebook 不会重训）。

训练日志通过 `python -u` 实时 flush 到 Colab cell。

In [ ]:
state_path = RUN_ROOT / 'trainer_state.json'
if state_path.exists():
    state = json.loads(state_path.read_text(encoding='utf-8'))
    current_step = int(state.get('env_step', 0))
else:
    current_step = 0
print(f'[state] cross run env_step = {current_step:,} / target {TOTAL_STEPS:,}')

if current_step >= TOTAL_STEPS:
    print(f'[skip] cross already extended to {current_step:,} ≥ {TOTAL_STEPS:,}')
elif current_step == 0:
    raise RuntimeError(
        f'no prior run at {RUN_ROOT}; this notebook expects a 600k checkpoint to resume from. '
        f'若需要从 0 开始,先跑 sac_arrival_v2_cross_u10_regression.ipynb'
    )
else:
    print(f'[train] resume from {current_step:,} → {TOTAL_STEPS:,} (extend +{TOTAL_STEPS - current_step:,})')
    !python -u -m scripts.train_sac \
        --resume {RUN_ROOT_STR} \
        --total-steps {TOTAL_STEPS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {MANIFEST_PATH_STR} \
        --device {DEVICE}

## 5. Phase 1 — Summarize: eval curve + final termination

重点看：
- 600k 之后 eval rows（[625k, 1M] 共 16 个 eval 点）是否收敛
- `last100k_mean` 是否 ≥ `0.9 × peak`（这次 last100k 跨 4 个 eval 点，比 600k 时方差小）
- `path_efficiency` / `eval_time` 是否仍在改进（如果 path_efficiency 在 800k 后饱和，说明真稳定了；如果还在涨，说明 1M 仍未收敛）

In [ ]:
eval_log_path = RUN_ROOT / 'results' / 'eval_log.csv'
final_eval_path = RUN_ROOT / 'results' / 'final_eval.json'
trainer_state_path = RUN_ROOT / 'trainer_state.json'

if not eval_log_path.exists():
    raise FileNotFoundError(f'missing eval log: {eval_log_path}')
if not final_eval_path.exists():
    raise FileNotFoundError(f'missing final eval: {final_eval_path}')

df = pd.read_csv(eval_log_path)
final_eval = json.loads(final_eval_path.read_text(encoding='utf-8'))
trainer_state = json.loads(trainer_state_path.read_text(encoding='utf-8')) if trainer_state_path.exists() else {}

peak_success = float(df['eval_success_rate'].max()) if len(df) else 0.0
peak_step = int(df.loc[df['eval_success_rate'].idxmax(), 'env_step']) if len(df) else 0
last100 = df[df['env_step'] >= TOTAL_STEPS - 100_000].copy()
last100_mean = float(last100['eval_success_rate'].mean()) if len(last100) else 0.0
final_success = float(final_eval['eval_success_rate'])
counts = final_eval.get('eval_termination_counts', {})
num_eps = float(final_eval.get('num_eval_episodes', EVAL_EPISODES))
oob_rate = float(counts.get('out_of_bounds', 0)) / max(num_eps, 1.0)

print('=' * 100)
print(f"{'metric':<32}{'value':>20}{'gate / note':>30}")
print('-' * 100)
print(f"{'final_success_rate':<32}{final_success:>20.4f}{f'>= {PASS_FINAL_SUCCESS:.2f}':>30}")
print(f"{'peak_success_rate':<32}{peak_success:>20.4f}{f'@ {peak_step:,}':>30}")
print(f"{'last100k_mean_success':<32}{last100_mean:>20.4f}{f'>= 0.9 * peak = {PASS_LAST100_RATIO * peak_success:.4f}':>30}")
print(f"{'final_oob_rate':<32}{oob_rate:>20.4f}{f'<= {PASS_OOB_RATE:.2f}':>30}")
print(f"{'obs_dim':<32}{trainer_state.get('observation_dim', 'NA'):>20}{'expected 48':>30}")
print(f"{'episode_context_obs':<32}{str(trainer_state.get('include_episode_context_obs', 'NA')):>20}{'expected True':>30}")
print(f"{'timeout_bootstrap':<32}{str(trainer_state.get('timeout_bootstrap_semantics', 'NA')):>20}{'expected terminal':>30}")
print('=' * 100)
print(f'termination_counts: {counts}')

print()
print('[last 16 eval rows — covers extension window 600k → 1M]')
cols = ['env_step', 'eval_success_rate', 'eval_return', 'eval_safety_cost', 'eval_time_s', 'eval_progress_ratio']
available = [c for c in cols if c in df.columns]
print(df[available].tail(16).to_string(index=False))

## 6. Phase 1 — Verdict: 是否允许进入 P1 v6

5 项 check 全过则 `READY_FOR_P1_V6 = True`。
Gate summary 落到 `results/cross_u10_extension_1M_gate_summary.json`，可与 600k 时的 `cross_u10_gate_summary.json` 并存。

In [ ]:
checks = [
    ('final success >= 0.85',
     final_success >= PASS_FINAL_SUCCESS,
     f'{final_success:.4f}'),
    ('last100k mean >= 0.9 * peak',
     last100_mean >= PASS_LAST100_RATIO * peak_success,
     f'{last100_mean:.4f} / peak={peak_success:.4f}'),
    ('final OOB rate <= 0.10',
     oob_rate <= PASS_OOB_RATE,
     f'{oob_rate:.4f}'),
    ('arrival_v2 context obs enabled',
     trainer_state.get('include_episode_context_obs') is True,
     str(trainer_state.get('include_episode_context_obs'))),
    ('arrival_v2 timeout terminal semantics',
     trainer_state.get('timeout_bootstrap_semantics') == 'terminal',
     str(trainer_state.get('timeout_bootstrap_semantics'))),
]

print('=' * 96)
print(f"{'check':<48}{'pass':>8}{'detail':>40}")
print('-' * 96)
all_pass = True
for name, ok, detail in checks:
    mark = 'PASS' if ok else 'FAIL'
    if not ok:
        all_pass = False
    print(f'{name:<48}{mark:>8}{detail:>40}')
print('=' * 96)
print()
READY_FOR_P1_V6 = bool(all_pass)
print(f'READY_FOR_P1_V6 = {READY_FOR_P1_V6}')

summary = {
    'phase': 'cross_u10_extension_1M',
    'benchmark': BENCHMARK_KEY,
    'objective': OBJECTIVE,
    'probe_layout': PROBE_LAYOUT,
    'history_length': HISTORY_LENGTH,
    'seed': SEED,
    'total_steps': TOTAL_STEPS,
    'final_success_rate': final_success,
    'peak_success_rate': peak_success,
    'peak_step': peak_step,
    'last100k_mean_success': last100_mean,
    'final_oob_rate': oob_rate,
    'termination_counts': counts,
    'ready_for_p1_v6': READY_FOR_P1_V6,
    'checks': [{'name': n, 'ok': bool(ok), 'detail': d} for n, ok, d in checks],
}
summary_path = RUN_ROOT / 'results' / 'cross_u10_extension_1M_gate_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(f'[saved] {summary_path}')

## 7. Phase 2 — P1 v6 配置 + manifest 生成

Phase 2 = `single_u15_upstream_tgt15` (s1 / 12-D / 1.5M)。这是真正暴露 efficiency_v2 旧 failure mode 的 benchmark；arrival_v2 是否修好 P1，本节给答案。

如果 `READY_FOR_P1_V6 == False`，本节仅打印状态，不生成 manifest，不训。

In [ ]:
print(f'READY_FOR_P1_V6 = {READY_FOR_P1_V6}')

if not READY_FOR_P1_V6:
    print('[hold] Phase 1 gate did not pass; Phase 2 stays held.')
    print('       回头看 last100k_mean / peak / OOB 哪条 fail,再决定是 reward 调参还是再延 budget')
else:
    print('[go] Phase 1 gate PASS — prepare Phase 2 manifest + P1 flow check')
    if not Path(P1_FLOW_PATH).exists():
        raise FileNotFoundError(f'missing P1 flow: {P1_FLOW_PATH}')
    if not P1_MANIFEST_PATH.exists():
        !python -u -m scripts.generate_standard_benchmarks --benchmarks {P1_BENCHMARK_KEY} --episodes {EVAL_EPISODES}
    if not P1_MANIFEST_PATH.exists():
        raise FileNotFoundError(f'P1 manifest still missing after generate: {P1_MANIFEST_PATH}')
    print(f'[OK] P1 flow exists: {P1_FLOW_PATH}')
    print(f'[OK] P1 manifest ready: {P1_MANIFEST_PATH}')

## 8. Phase 2 — Run P1 v6 (1.5M)

Skip / resume 逻辑：
- `env_step >= P1_TOTAL_STEPS` → skip
- `0 < env_step < P1_TOTAL_STEPS` → 用 `--resume` 续训（中断重连场景）
- `env_step == 0` → 全量 fresh start

训练日志同样 `python -u` 实时输出，不再用 `%%bash`。

**执行时间**：1.5M × num_envs=6 × s1 obs 12-D ≈ 5-6h L4。Colab Pro 单 session 建议在中点保存 checkpoint（已设 `CHECKPOINT_EVERY=100k`），断线后本 cell 重跑会自动 resume。

In [ ]:
if not READY_FOR_P1_V6:
    print('[hold] Phase 1 gate did not pass; skip Phase 2 training entirely')
else:
    p1_state_path = P1_RUN_ROOT / 'trainer_state.json'
    if p1_state_path.exists():
        p1_state = json.loads(p1_state_path.read_text(encoding='utf-8'))
        p1_current_step = int(p1_state.get('env_step', 0))
    else:
        p1_current_step = 0
    print(f'[state] P1 env_step = {p1_current_step:,} / target {P1_TOTAL_STEPS:,}')

    if p1_current_step >= P1_TOTAL_STEPS:
        print(f'[skip] P1 already trained to {p1_current_step:,} ≥ {P1_TOTAL_STEPS:,}')
    elif p1_current_step > 0:
        print(f'[resume] P1 continuing from {p1_current_step:,} → {P1_TOTAL_STEPS:,}')
        !python -u -m scripts.train_sac \
            --resume {P1_RUN_ROOT_STR} \
            --total-steps {P1_TOTAL_STEPS} \
            --eval-every {EVAL_EVERY} \
            --eval-episodes {EVAL_EPISODES} \
            --checkpoint-every {CHECKPOINT_EVERY} \
            --eval-manifest {P1_MANIFEST_PATH_STR} \
            --device {DEVICE}
    else:
        print(f'[train] P1 fresh start → {P1_TOTAL_STEPS:,}')
        !python -u -m scripts.train_sac \
            --flow {P1_FLOW_PATH} \
            --task-geometry {P1_TASK_GEOMETRY} \
            --target-speed {TARGET_SPEED} \
            --objective {OBJECTIVE} \
            --probe-layout {P1_PROBE_LAYOUT} \
            --history-length {HISTORY_LENGTH} \
            --total-steps {P1_TOTAL_STEPS} \
            --random-steps {RANDOM_STEPS} \
            --update-after {UPDATE_AFTER} \
            --batch-size {BATCH_SIZE} \
            --hidden-dim {HIDDEN_DIM} \
            --num-envs {NUM_ENVS} \
            --eval-every {EVAL_EVERY} \
            --eval-episodes {EVAL_EPISODES} \
            --checkpoint-every {CHECKPOINT_EVERY} \
            --eval-manifest {P1_MANIFEST_PATH_STR} \
            --seed {SEED} \
            --device {DEVICE} \
            --save-dir {P1_RUN_ROOT_STR} \
            --checkpoint-dir {P1_CKPT_ROOT_STR}

## 9. Phase 2 — Summarize: P1 v6 final eval + curve

若 Phase 1 hold，直接打印 hold 状态。

P1 v6 关注点：
- `final_success_rate` 是否 ≥ 0.85（arrival_v2 修好 efficiency_v2 旧 failure mode 的核心证据）
- `last100k_mean` ≥ 0.9 × peak 是否成立（与 cross 同一稳定性判据）
- termination 分布：goal vs timeout vs out_of_bounds 各占多少（P1 旧版的核心问题就是 success 退化到 0、return 上升）
- 如果 1.5M 仍未收敛但仍在上升，再决定是否 extend 到 2M

In [ ]:
if not READY_FOR_P1_V6:
    print('[hold] Phase 1 gate did not pass; no P1 results to summarize.')
else:
    p1_eval_log = P1_RUN_ROOT / 'results' / 'eval_log.csv'
    p1_final_eval = P1_RUN_ROOT / 'results' / 'final_eval.json'
    p1_trainer_state = P1_RUN_ROOT / 'trainer_state.json'

    if not p1_final_eval.exists():
        raise FileNotFoundError(f'missing P1 final eval: {p1_final_eval}')

    p1_df = pd.read_csv(p1_eval_log) if p1_eval_log.exists() else pd.DataFrame()
    p1_final = json.loads(p1_final_eval.read_text(encoding='utf-8'))
    p1_state_d = json.loads(p1_trainer_state.read_text(encoding='utf-8')) if p1_trainer_state.exists() else {}
    p1_counts = p1_final.get('eval_termination_counts', {})

    p1_peak = float(p1_df['eval_success_rate'].max()) if len(p1_df) else 0.0
    p1_peak_step = int(p1_df.loc[p1_df['eval_success_rate'].idxmax(), 'env_step']) if len(p1_df) else 0
    p1_last100 = p1_df[p1_df['env_step'] >= P1_TOTAL_STEPS - 100_000] if len(p1_df) else p1_df
    p1_last100_mean = float(p1_last100['eval_success_rate'].mean()) if len(p1_last100) else 0.0

    print('=' * 96)
    print('P1_V6_FINAL  (arrival_v2 / s1 / upstream / 1.5M)')
    print('-' * 96)
    print(f"  final_success_rate    : {p1_final['eval_success_rate']:.4f}")
    print(f"  peak_success_rate     : {p1_peak:.4f}  @ {p1_peak_step:,}")
    print(f"  last100k_mean_success : {p1_last100_mean:.4f}  (gate {PASS_LAST100_RATIO * p1_peak:.4f})")
    print(f"  avg_return            : {p1_final.get('eval_return', 0):.2f}")
    print(f"  avg_time_s            : {p1_final.get('eval_time_s', 0):.2f}")
    print(f"  progress_ratio        : {p1_final.get('eval_progress_ratio', 0):.4f}")
    print(f"  safety_cost           : {p1_final.get('eval_safety_cost', 0):.4f}")
    print(f"  termination           : {p1_counts}")
    print(f"  obs_dim               : {p1_state_d.get('observation_dim', 'NA')}")
    print(f"  context_obs           : {p1_state_d.get('include_episode_context_obs', 'NA')}")
    print(f"  timeout_bootstrap     : {p1_state_d.get('timeout_bootstrap_semantics', 'NA')}")
    print('=' * 96)

    if len(p1_df):
        print()
        print('[last 16 eval rows]')
        cols = ['env_step', 'eval_success_rate', 'eval_return', 'eval_safety_cost',
                'eval_time_s', 'eval_progress_ratio']
        available = [c for c in cols if c in p1_df.columns]
        print(p1_df[available].tail(16).to_string(index=False))